# Battery-storage economics — Haslach

This notebook reconstructs unmanaged heating demand and estimates the technical and financial effect of an AC-coupled, self-consumption battery. It is analysis-only and does not alter either runtime or the source database.

**Sign convention:** positive `power_mains` is grid import; negative is export. Generation and load columns are positive. The myPV unit diverts solar surplus into **space heating**; the BWWP electric element diverts solar surplus into **domestic hot water**. The counterfactual preserves both thermal services by replacing myPV energy with main-heat-pump electricity and BWWP element energy with compressor electricity, using the configured COP.

The available data covers only part of a year. Observed-period results are valid for the retained data, but lifecycle economics are withheld unless enough eligible days are present. Explicit partial-year annualization is provisional and seasonally biased, and is not suitable for an investment decision.


In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ANALYSIS_START = "2026-07-01"
ANALYSIS_END = None
RESAMPLE_FREQUENCY = "5min"
MINIMUM_DAILY_COVERAGE = 0.95
HEAT_PUMP_COP = 4.0
HEAT_PUMP_STANDBY_THRESHOLD_W = 100.0

BATTERY_CAPACITY_KWH = 10.0
INITIAL_SOC_FRACTION = 0.5
MAX_CHARGE_POWER_KW = 5.0
MAX_DISCHARGE_POWER_KW = 5.0
CHARGE_EFFICIENCY = 0.95
DISCHARGE_EFFICIENCY = 0.95
ANNUAL_CAPACITY_DEGRADATION = 0.02

ELECTRICITY_PRICE_EUR_PER_KWH = 0.30
FEED_IN_TARIFF_EUR_PER_KWH = 0.08
INSTALLED_BATTERY_COST_EUR = 8000.0
ANNUAL_OM_COST_EUR = 50.0
PROJECT_LIFETIME_YEARS = 15
DISCOUNT_RATE = 0.04
MINIMUM_DAYS_FOR_LIFECYCLE = 330
ALLOW_PARTIAL_YEAR_ANNUALIZATION = False
SENSITIVITY_CAPACITIES_KWH = [0.0, 5.0, 7.5, 10.0, 12.5, 15.0, 20.0]

ASSUMPTIONS = {
    "capacity_kwh": BATTERY_CAPACITY_KWH,
    "initial_soc_fraction": INITIAL_SOC_FRACTION,
    "max_charge_power_kw": MAX_CHARGE_POWER_KW,
    "max_discharge_power_kw": MAX_DISCHARGE_POWER_KW,
    "charge_efficiency": CHARGE_EFFICIENCY,
    "discharge_efficiency": DISCHARGE_EFFICIENCY,
    "annual_capacity_degradation": ANNUAL_CAPACITY_DEGRADATION,
    "electricity_price": ELECTRICITY_PRICE_EUR_PER_KWH,
    "feed_in_tariff": FEED_IN_TARIFF_EUR_PER_KWH,
    "installed_cost": INSTALLED_BATTERY_COST_EUR,
    "annual_om_cost": ANNUAL_OM_COST_EUR,
    "project_lifetime_years": PROJECT_LIFETIME_YEARS,
    "discount_rate": DISCOUNT_RATE,
    "minimum_days_for_lifecycle": MINIMUM_DAYS_FOR_LIFECYCLE,
}
REQUIRED_COLUMNS = [
    "timestamp", "power_mains", "power_pv", "power_mypv", "power_bwwp",
    "power_wp", "controller_state", "fhs280_compressor", "fhs280_elpatron",
]


## Synthetic replacement-allocation checks

The fixture has one complete day and one deliberately incomplete day. Its two managed solar-thermal loads are tested independently: myPV space-heating energy must be replaced by the main HP, and BWWP EL hot-water energy by the BWWP compressor profile.


In [ ]:
fixture_times = pd.date_range("2026-07-01", periods=288, freq="5min", tz="Europe/Berlin")
synthetic_source = pd.DataFrame({
    "timestamp": fixture_times.append(fixture_times[:120] + pd.Timedelta(days=1)),
    "power_mains": 1000.0,
    "power_pv": 2500.0,
    "power_mypv": 400.0,
    "power_bwwp": 800.0,
    "power_wp": 1200.0,
    "controller_state": 1,
    "fhs280_compressor": 1,
    "fhs280_elpatron": 0,
})
synthetic_source.loc[synthetic_source.index < 60, "fhs280_elpatron"] = 1

def _interval_hours(frame):
    if "interval_hours" in frame.attrs:
        return float(frame.attrs["interval_hours"])
    differences = frame.index.to_series().diff().dropna()
    if differences.empty:
        raise ValueError("Cannot infer interval duration from fewer than two timestamps")
    return differences.median().total_seconds() / 3600.0

def prepare_intervals(source, frequency, minimum_daily_coverage):
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(source.columns))
    if missing_columns:
        raise ValueError(f"Missing source columns: {missing_columns}")
    data = source[REQUIRED_COLUMNS].copy()
    data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
    if data["timestamp"].dt.tz is None:
        data["timestamp"] = data["timestamp"].dt.tz_localize("Europe/Berlin")
    else:
        data["timestamp"] = data["timestamp"].dt.tz_convert("Europe/Berlin")
    data = data.dropna(subset=["timestamp"]).sort_values("timestamp")
    data = data.drop_duplicates(subset="timestamp", keep="last")
    if ANALYSIS_START is not None:
        start = pd.Timestamp(ANALYSIS_START, tz="Europe/Berlin")
        data = data.loc[data["timestamp"] >= start]
    if ANALYSIS_END is not None:
        end_exclusive = pd.Timestamp(ANALYSIS_END, tz="Europe/Berlin") + pd.DateOffset(days=1)
        data = data.loc[data["timestamp"] < end_exclusive]
    if data.empty:
        raise ValueError("No source samples remain inside the configured analysis range")

    numeric_columns = [column for column in REQUIRED_COLUMNS if column != "timestamp"]
    data[numeric_columns] = data[numeric_columns].apply(pd.to_numeric, errors="coerce")
    power_columns = ["power_mains", "power_pv", "power_mypv", "power_bwwp", "power_wp"]
    state_columns = ["controller_state", "fhs280_compressor", "fhs280_elpatron"]
    indexed = data.set_index("timestamp")
    power = indexed[power_columns].resample(frequency, origin="start_day").mean()
    states = indexed[state_columns].resample(frequency, origin="start_day").last()
    first_day = indexed.index.min().normalize()
    last_day = indexed.index.max().normalize()
    if ANALYSIS_START is not None:
        first_day = min(first_day, pd.Timestamp(ANALYSIS_START, tz="Europe/Berlin").normalize())
    if ANALYSIS_END is not None:
        last_day = max(last_day, pd.Timestamp(ANALYSIS_END, tz="Europe/Berlin").normalize())
    grid_start = first_day
    grid_end = last_day + pd.DateOffset(days=1)
    full_index = pd.date_range(grid_start, grid_end, freq=frequency, inclusive="left")
    intervals = pd.concat([power, states], axis=1).reindex(full_index)
    intervals.index.name = "timestamp"

    valid = intervals[["power_mains", "power_pv"]].notna().all(axis=1)
    daily_quality_rows = []
    for day in pd.date_range(first_day, last_day, freq="D"):
        day_index = full_index[full_index.normalize() == day]
        valid_count = int(valid.reindex(day_index, fill_value=False).sum())
        expected_count = len(day_index)
        coverage = valid_count / expected_count if expected_count else 0.0
        daily_quality_rows.append({
            "date": day,
            "expected_intervals": expected_count,
            "valid_intervals": valid_count,
            "coverage": coverage,
            "eligible": coverage >= minimum_daily_coverage,
        })
    daily_quality = pd.DataFrame(daily_quality_rows).set_index("date")
    eligible_days = daily_quality.index[daily_quality["eligible"]]
    intervals = intervals.loc[valid & intervals.index.normalize().isin(eligible_days)].copy()
    interval_delta = pd.Timedelta(frequency)
    discontinuity = intervals.index.to_series().diff().gt(interval_delta)
    intervals["segment_id"] = discontinuity.cumsum().astype(int).to_numpy()
    intervals.attrs["interval_hours"] = interval_delta.total_seconds() / 3600.0
    return intervals, daily_quality

def build_heating_counterfactual(intervals, cop, standby_threshold_w):
    if cop <= 0:
        raise ValueError("COP must be greater than zero")
    frame = intervals.copy()
    interval_hours = _interval_hours(frame)
    frame["power_mypv"] = pd.to_numeric(frame["power_mypv"], errors="coerce").fillna(0.0)
    frame["power_bwwp"] = pd.to_numeric(frame["power_bwwp"], errors="coerce").fillna(0.0)
    frame["power_wp"] = pd.to_numeric(frame["power_wp"], errors="coerce").fillna(0.0)
    frame["observed_load"] = frame["power_pv"] + frame["power_mains"]
    frame["bwwp_el_power"] = frame["power_bwwp"].where(
        frame["fhs280_elpatron"].eq(1), 0.0
    ).clip(lower=0.0)
    frame["mypv_replacement_hp_power"] = 0.0
    frame["bwwp_replacement_hp_power"] = 0.0
    fallback_rows = []
    negative_interval_count = 0
    negative_energy_wh = 0.0

    for day in frame.index.normalize().unique():
        day_mask = frame.index.normalize() == day
        day_frame = frame.loc[day_mask]
        mypv_energy_wh = day_frame["power_mypv"].clip(lower=0.0).sum() * interval_hours
        bwwp_energy_wh = day_frame["bwwp_el_power"].sum() * interval_hours
        mypv_target_wh = mypv_energy_wh / cop
        bwwp_target_wh = bwwp_energy_wh / cop

        mypv_weights = day_frame["power_wp"].where(
            day_frame["power_wp"] > standby_threshold_w, 0.0
        ).clip(lower=0.0)
        bwwp_weights = day_frame["power_bwwp"].where(
            day_frame["fhs280_compressor"].eq(1)
            & day_frame["fhs280_elpatron"].ne(1), 0.0
        ).clip(lower=0.0)

        for source_name, target_wh, weights, output_column in (
            ("myPV_space_heating", mypv_target_wh, mypv_weights, "mypv_replacement_hp_power"),
            ("BWWP_hot_water", bwwp_target_wh, bwwp_weights, "bwwp_replacement_hp_power"),
        ):
            if target_wh <= 0.0:
                continue
            total_weight = float(weights.sum())
            if total_weight <= 0.0:
                weights = pd.Series(1.0, index=day_frame.index)
                total_weight = float(len(weights))
                fallback_rows.append({
                    "source": source_name,
                    "date": day,
                    "reason": "no positive empirical replacement profile; allocated equally",
                })
            frame.loc[day_mask, output_column] = (
                target_wh * weights.to_numpy() / total_weight / interval_hours
            )

        raw_load = (
            day_frame["observed_load"]
            - day_frame["power_mypv"]
            - day_frame["bwwp_el_power"]
            + frame.loc[day_mask, "mypv_replacement_hp_power"]
            + frame.loc[day_mask, "bwwp_replacement_hp_power"]
        )
        negative = raw_load < 0.0
        negative_interval_count += int(negative.sum())
        negative_energy_wh += float((-raw_load.loc[negative]).sum() * interval_hours)
        frame.loc[day_mask, "counterfactual_load_raw"] = raw_load

    frame["counterfactual_load"] = frame["counterfactual_load_raw"].clip(lower=0.0)
    frame["counterfactual_mains"] = frame["counterfactual_load"] - frame["power_pv"]
    frame.attrs["interval_hours"] = interval_hours
    frame.attrs["clipped_negative_interval_count"] = negative_interval_count
    frame.attrs["clipped_negative_energy_wh"] = negative_energy_wh
    return frame, pd.DataFrame(fallback_rows, columns=["source", "date", "reason"])

prepared, daily_quality = prepare_intervals(
    synthetic_source, frequency="5min", minimum_daily_coverage=0.95
)
counterfactual, fallback_days = build_heating_counterfactual(
    prepared, cop=4.0, standby_threshold_w=100.0
)
assert daily_quality.loc[daily_quality.index[0], "eligible"]
assert not daily_quality.loc[daily_quality.index[1], "eligible"]
assert np.isclose(
    counterfactual["mypv_replacement_hp_power"].sum(),
    counterfactual["power_mypv"].sum() / 4.0,
)
assert np.isclose(
    counterfactual["bwwp_replacement_hp_power"].sum(),
    counterfactual["bwwp_el_power"].sum() / 4.0,
)

fallback_source = synthetic_source.copy()
fallback_source["power_wp"] = 0.0
fallback_prepared, _ = prepare_intervals(
    fallback_source, frequency="5min", minimum_daily_coverage=0.95
)
fallback_counterfactual, synthetic_fallback_days = build_heating_counterfactual(
    fallback_prepared, cop=4.0, standby_threshold_w=100.0
)
assert "myPV_space_heating" in set(synthetic_fallback_days["source"])
assert "BWWP_hot_water" not in set(synthetic_fallback_days["source"])
assert fallback_counterfactual["mypv_replacement_hp_power"].nunique() == 1
assert np.isclose(
    fallback_counterfactual["mypv_replacement_hp_power"].sum(),
    fallback_counterfactual["power_mypv"].sum() / 4.0,
)

dispatch_fixture = pd.DataFrame(
    {"counterfactual_mains": [-4000.0, -4000.0, 3000.0, 3000.0], "segment_id": 0},
    index=pd.date_range("2026-07-01", periods=4, freq="h", tz="Europe/Berlin"),
)
dispatch_fixture.attrs["interval_hours"] = 1.0

def simulate_battery(
    frame, capacity_kwh, initial_soc_fraction, max_charge_power_kw,
    max_discharge_power_kw, charge_efficiency, discharge_efficiency,
):
    if capacity_kwh < 0.0:
        raise ValueError("Battery capacity cannot be negative")
    if not 0.0 <= initial_soc_fraction <= 1.0:
        raise ValueError("Initial SOC fraction must be between zero and one")
    if max_charge_power_kw < 0.0 or max_discharge_power_kw < 0.0:
        raise ValueError("Charge and discharge power limits cannot be negative")
    if not 0.0 < charge_efficiency <= 1.0 or not 0.0 < discharge_efficiency <= 1.0:
        raise ValueError("Battery efficiencies must be in (0, 1]")
    if frame.empty or "segment_id" not in frame or "counterfactual_mains" not in frame:
        raise ValueError("Dispatch requires intervals, segment_id, and counterfactual_mains")
    baseline = pd.to_numeric(frame["counterfactual_mains"], errors="coerce").to_numpy(dtype=float)
    if not np.isfinite(baseline).all() or frame["segment_id"].isna().any():
        raise ValueError("Dispatch inputs must have finite mains power and segment identifiers")
    interval_hours = _interval_hours(frame)
    charge_limit_w = max_charge_power_kw * 1000.0
    discharge_limit_w = max_discharge_power_kw * 1000.0
    charge = np.zeros(len(frame), dtype=float)
    discharge = np.zeros(len(frame), dtype=float)
    soc = np.zeros(len(frame), dtype=float)
    resets = np.zeros(len(frame), dtype=bool)
    charge_loss = np.zeros(len(frame), dtype=float)
    discharge_loss = np.zeros(len(frame), dtype=float)
    charge_power_limited = np.zeros(len(frame), dtype=float)
    charge_capacity_limited = np.zeros(len(frame), dtype=float)
    discharge_power_limited = np.zeros(len(frame), dtype=float)
    discharge_energy_limited = np.zeros(len(frame), dtype=float)
    stored_kwh = 0.0
    previous_segment = object()

    for position, (segment, mains_w) in enumerate(zip(frame["segment_id"].to_numpy(), baseline)):
        if position == 0 or segment != previous_segment:
            stored_kwh = capacity_kwh * initial_soc_fraction
            resets[position] = True
        if mains_w < 0.0:
            available_export_w = -mains_w
            power_available_w = min(available_export_w, charge_limit_w)
            charge_power_limited[position] = available_export_w - power_available_w
            capacity_available_w = max(
                (capacity_kwh - stored_kwh) / (charge_efficiency * interval_hours) * 1000.0,
                0.0,
            )
            charge[position] = min(power_available_w, capacity_available_w)
            charge_capacity_limited[position] = power_available_w - charge[position]
            stored_added_kwh = charge[position] * interval_hours / 1000.0 * charge_efficiency
            charge_loss[position] = charge[position] * interval_hours * (1.0 - charge_efficiency)
            stored_kwh += stored_added_kwh
        elif mains_w > 0.0:
            available_import_w = mains_w
            power_available_w = min(available_import_w, discharge_limit_w)
            discharge_power_limited[position] = available_import_w - power_available_w
            energy_available_w = stored_kwh * discharge_efficiency / interval_hours * 1000.0
            discharge[position] = min(power_available_w, energy_available_w)
            discharge_energy_limited[position] = power_available_w - discharge[position]
            stored_removed_kwh = discharge[position] * interval_hours / 1000.0 / discharge_efficiency
            discharge_loss[position] = (
                stored_removed_kwh - discharge[position] * interval_hours / 1000.0
            ) * 1000.0
            stored_kwh -= stored_removed_kwh
        stored_kwh = min(max(stored_kwh, 0.0), capacity_kwh)
        soc[position] = stored_kwh
        previous_segment = segment

    result = frame.copy()
    result["battery_charge_power_w"] = charge
    result["battery_discharge_power_w"] = discharge
    result["soc_kwh"] = soc
    result["soc_reset"] = resets
    result["charge_loss_wh"] = charge_loss
    result["discharge_loss_wh"] = discharge_loss
    result["post_battery_mains"] = baseline + charge - discharge
    result["residual_import_power_w"] = result["post_battery_mains"].clip(lower=0.0)
    result["residual_export_power_w"] = (-result["post_battery_mains"]).clip(lower=0.0)
    result["charge_power_limited_w"] = charge_power_limited
    result["charge_capacity_limited_w"] = charge_capacity_limited
    result["discharge_power_limited_w"] = discharge_power_limited
    result["discharge_energy_limited_w"] = discharge_energy_limited
    result.attrs["interval_hours"] = interval_hours
    result.attrs["capacity_kwh"] = capacity_kwh
    result.attrs["charge_efficiency"] = charge_efficiency
    result.attrs["discharge_efficiency"] = discharge_efficiency
    return result

dispatch_test = simulate_battery(
    dispatch_fixture,
    capacity_kwh=5.0,
    initial_soc_fraction=0.0,
    max_charge_power_kw=3.0,
    max_discharge_power_kw=3.0,
    charge_efficiency=0.9,
    discharge_efficiency=0.9,
)
assert np.all(dispatch_test["battery_charge_power_w"] >= 0)
assert np.all(dispatch_test["battery_discharge_power_w"] >= 0)
assert not ((dispatch_test["battery_charge_power_w"] > 0) & (dispatch_test["battery_discharge_power_w"] > 0)).any()
assert dispatch_test["soc_kwh"].between(0.0, 5.0).all()
assert np.isclose(dispatch_test.iloc[0]["battery_charge_power_w"], 3000.0)
assert np.isclose(dispatch_test.iloc[0]["soc_kwh"], 2.7)
assert np.isclose(dispatch_test.iloc[2]["battery_discharge_power_w"], 3000.0)
assert dispatch_test["soc_reset"].tolist() == [True, False, False, False]

reset_fixture = dispatch_fixture.copy()
reset_fixture["segment_id"] = [0, 0, 1, 1]
reset_dispatch = simulate_battery(
    reset_fixture,
    capacity_kwh=5.0,
    initial_soc_fraction=0.0,
    max_charge_power_kw=3.0,
    max_discharge_power_kw=3.0,
    charge_efficiency=0.9,
    discharge_efficiency=0.9,
)
assert reset_dispatch["soc_reset"].tolist() == [True, False, True, False]
assert np.isclose(reset_dispatch.iloc[2]["battery_discharge_power_w"], 0.0)


## Read-only source data and interval quality

The local integrity-checked snapshot is preferred when present; otherwise the standard `database/haslach.db` input is used. SQLite is opened in read-only mode. Analysis dates are interpreted as local `Europe/Berlin` calendar dates.

In [ ]:
SNAPSHOT_FILENAME = "haslach-stabilization-2026-10-04.db"

def find_db_path():
    roots = [Path.cwd(), *Path.cwd().parents]
    candidates = [root / ".tmp_docs" / SNAPSHOT_FILENAME for root in roots]
    candidates += [root / "database" / "haslach.db" for root in roots]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    searched = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"No Haslach database found; searched: {searched}")

def load_source_data(db_path):
    db_path = Path(db_path).resolve()
    params = []
    predicates = []
    if ANALYSIS_START is not None:
        start_utc = pd.Timestamp(ANALYSIS_START, tz="Europe/Berlin").tz_convert("UTC").isoformat()
        predicates.append('julianday("timestamp") >= julianday(?)')
        params.append(start_utc)
    if ANALYSIS_END is not None:
        end_exclusive = pd.Timestamp(ANALYSIS_END, tz="Europe/Berlin") + pd.DateOffset(days=1)
        end_utc = end_exclusive.tz_convert("UTC").isoformat()
        predicates.append('julianday("timestamp") < julianday(?)')
        params.append(end_utc)
    columns = ", ".join(f'"{column}"' for column in REQUIRED_COLUMNS)
    query = f"SELECT {columns} FROM main"
    if predicates:
        query += " WHERE " + " AND ".join(predicates)
    query += ' ORDER BY "timestamp"'
    uri = db_path.as_uri() + "?mode=ro"
    with sqlite3.connect(uri, uri=True) as connection:
        data = pd.read_sql_query(query, connection, params=params)
    if data.empty:
        raise ValueError(f"No rows found in {db_path} for the configured analysis dates")
    data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True).dt.tz_convert("Europe/Berlin")
    return data


In [ ]:
db_path = find_db_path()
source_data = load_source_data(db_path)
intervals, daily_quality = prepare_intervals(
    source_data,
    frequency=RESAMPLE_FREQUENCY,
    minimum_daily_coverage=MINIMUM_DAILY_COVERAGE,
)
counterfactual, fallback_days = build_heating_counterfactual(
    intervals,
    cop=HEAT_PUMP_COP,
    standby_threshold_w=HEAT_PUMP_STANDBY_THRESHOLD_W,
)
interval_hours = _interval_hours(counterfactual)
source_spacing_minutes = source_data["timestamp"].diff().dt.total_seconds().div(60).dropna()

print(f"Read-only database: {db_path}")
print(f"Source date range: {source_data['timestamp'].min()} — {source_data['timestamp'].max()}")
print(f"Retained interval range: {counterfactual.index.min()} — {counterfactual.index.max()}")
print(f"Source rows: {len(source_data):,}; retained intervals: {len(counterfactual):,}")
print("Missing required source values:")
display(source_data[REQUIRED_COLUMNS].isna().sum().rename("missing_rows").to_frame())
print("Source timestamp spacing (minutes):")
display(source_spacing_minutes.describe().rename("minutes").to_frame())
print("Daily data quality:")
display(daily_quality)
print("Excluded days:")
display(daily_quality.loc[~daily_quality["eligible"]])
print("Heating-profile fallback days:")
display(fallback_days)
print(
    "Clipped reconstructed load: "
    f"{counterfactual.attrs['clipped_negative_interval_count']} intervals, "
    f"{counterfactual.attrs['clipped_negative_energy_wh']:.3f} Wh below zero"
)

replacement_energy_checks_pass = True
for day in counterfactual.index.normalize().unique():
    day_frame = counterfactual.loc[counterfactual.index.normalize() == day]
    mypv_replaced_wh = day_frame["mypv_replacement_hp_power"].sum() * interval_hours
    mypv_target_wh = day_frame["power_mypv"].clip(lower=0.0).sum() * interval_hours / HEAT_PUMP_COP
    bwwp_replaced_wh = day_frame["bwwp_replacement_hp_power"].sum() * interval_hours
    bwwp_target_wh = day_frame["bwwp_el_power"].sum() * interval_hours / HEAT_PUMP_COP
    replacement_energy_checks_pass &= np.isclose(mypv_replaced_wh, mypv_target_wh, rtol=1e-10, atol=1e-8)
    replacement_energy_checks_pass &= np.isclose(bwwp_replaced_wh, bwwp_target_wh, rtol=1e-10, atol=1e-8)
assert replacement_energy_checks_pass

dispatch = simulate_battery(
    counterfactual,
    capacity_kwh=BATTERY_CAPACITY_KWH,
    initial_soc_fraction=INITIAL_SOC_FRACTION,
    max_charge_power_kw=MAX_CHARGE_POWER_KW,
    max_discharge_power_kw=MAX_DISCHARGE_POWER_KW,
    charge_efficiency=CHARGE_EFFICIENCY,
    discharge_efficiency=DISCHARGE_EFFICIENCY,
)
maximum_dispatch_balance_residual_w = float(
    (dispatch["post_battery_mains"] - (
        dispatch["counterfactual_mains"]
        + dispatch["battery_charge_power_w"]
        - dispatch["battery_discharge_power_w"]
    )).abs().max()
)
assert dispatch["soc_kwh"].between(-1e-9, BATTERY_CAPACITY_KWH + 1e-9).all()
assert not ((dispatch["battery_charge_power_w"] > 1e-9) & (dispatch["battery_discharge_power_w"] > 1e-9)).any()
assert not ((dispatch["battery_charge_power_w"] > 1e-9) & (dispatch["counterfactual_mains"] > 1e-9)).any()
assert not ((dispatch["battery_discharge_power_w"] > 1e-9) & (dispatch["counterfactual_mains"] < -1e-9)).any()
assert maximum_dispatch_balance_residual_w < 1e-6
print(f"Maximum dispatch balance residual: {maximum_dispatch_balance_residual_w:.3g} W")
print(f"SOC reset boundaries: {int(dispatch['soc_reset'].sum())}")


## Synthetic energy and financial checks

These checks verify grid-energy reconciliation including conversion losses, observed-period bill savings, year-zero/year-one cash flow, and the lifecycle guard for insufficient data.

In [ ]:
def summarize_energy(dispatch):
    interval_hours = _interval_hours(dispatch)
    baseline_mains = dispatch["counterfactual_mains"]
    post_mains = dispatch["post_battery_mains"]
    pv_power = dispatch["power_pv"] if "power_pv" in dispatch else pd.Series(0.0, index=dispatch.index)
    demand_power = (
        dispatch["counterfactual_load"]
        if "counterfactual_load" in dispatch
        else pd.Series(0.0, index=dispatch.index)
    )
    to_kwh = interval_hours / 1000.0
    baseline_import = baseline_mains.clip(lower=0.0).sum() * to_kwh
    baseline_export = (-baseline_mains).clip(lower=0.0).sum() * to_kwh
    post_import = post_mains.clip(lower=0.0).sum() * to_kwh
    post_export = (-post_mains).clip(lower=0.0).sum() * to_kwh
    pv_generation = pv_power.clip(lower=0.0).sum() * to_kwh
    counterfactual_demand = demand_power.clip(lower=0.0).sum() * to_kwh
    charge_kwh = dispatch["battery_charge_power_w"].sum() * to_kwh
    discharge_kwh = dispatch["battery_discharge_power_w"].sum() * to_kwh
    charge_efficiency = float(dispatch.attrs.get("charge_efficiency", CHARGE_EFFICIENCY))
    discharge_efficiency = float(dispatch.attrs.get("discharge_efficiency", DISCHARGE_EFFICIENCY))
    stored_change_kwh = charge_kwh * charge_efficiency - discharge_kwh / discharge_efficiency
    charge_losses_kwh = dispatch["charge_loss_wh"].sum() / 1000.0
    discharge_losses_kwh = dispatch["discharge_loss_wh"].sum() / 1000.0
    total_losses_kwh = charge_losses_kwh + discharge_losses_kwh
    baseline_net_grid = baseline_import - baseline_export
    post_net_grid = post_import - post_export
    balance_residual = (post_net_grid - baseline_net_grid) - stored_change_kwh - total_losses_kwh
    capacity_kwh = float(dispatch.attrs.get("capacity_kwh", BATTERY_CAPACITY_KWH))
    baseline_pv_self_consumption = 1.0 - baseline_export / pv_generation if pv_generation > 0.0 else np.nan
    pv_self_consumption = 1.0 - post_export / pv_generation if pv_generation > 0.0 else np.nan
    baseline_self_sufficiency = 1.0 - baseline_import / counterfactual_demand if counterfactual_demand > 0.0 else np.nan
    self_sufficiency = 1.0 - post_import / counterfactual_demand if counterfactual_demand > 0.0 else np.nan
    return pd.Series({
        "baseline_import_kwh": baseline_import,
        "baseline_export_kwh": baseline_export,
        "post_battery_import_kwh": post_import,
        "post_battery_export_kwh": post_export,
        "baseline_net_grid_kwh": baseline_net_grid,
        "post_net_grid_kwh": post_net_grid,
        "pv_generation_kwh": pv_generation,
        "counterfactual_demand_kwh": counterfactual_demand,
        "battery_charge_kwh": charge_kwh,
        "battery_discharge_kwh": discharge_kwh,
        "stored_energy_change_kwh": stored_change_kwh,
        "charge_losses_kwh": charge_losses_kwh,
        "discharge_losses_kwh": discharge_losses_kwh,
        "conversion_losses_kwh": total_losses_kwh,
        "energy_balance_residual_kwh": balance_residual,
        "equivalent_full_cycles": charge_kwh * charge_efficiency / capacity_kwh if capacity_kwh > 0 else np.nan,
        "peak_charge_power_kw": dispatch["battery_charge_power_w"].max() / 1000.0,
        "peak_discharge_power_kw": dispatch["battery_discharge_power_w"].max() / 1000.0,
        "peak_pv_power_kw": pv_power.max() / 1000.0,
        "final_soc_kwh": dispatch["soc_kwh"].iloc[-1],
        "soc_reset_count": int(dispatch["soc_reset"].sum()),
        "soc_after_reset_kwh": tuple(dispatch.loc[dispatch["soc_reset"], "soc_kwh"].tolist()),
        "baseline_pv_self_consumption": baseline_pv_self_consumption,
        "pv_self_consumption": pv_self_consumption,
        "pv_self_consumption_change": pv_self_consumption - baseline_pv_self_consumption,
        "baseline_self_sufficiency": baseline_self_sufficiency,
        "self_sufficiency": self_sufficiency,
        "self_sufficiency_change": self_sufficiency - baseline_self_sufficiency,
    })

def calculate_observed_savings(dispatch, electricity_price, feed_in_tariff):
    interval_hours = _interval_hours(dispatch)
    to_kwh = interval_hours / 1000.0
    baseline = dispatch["counterfactual_mains"]
    post = dispatch["post_battery_mains"]
    monthly = pd.DataFrame(index=dispatch.index)
    monthly["baseline_import_kwh"] = baseline.clip(lower=0.0) * to_kwh
    monthly["baseline_export_kwh"] = (-baseline).clip(lower=0.0) * to_kwh
    monthly["post_battery_import_kwh"] = post.clip(lower=0.0) * to_kwh
    monthly["post_battery_export_kwh"] = (-post).clip(lower=0.0) * to_kwh
    monthly["baseline_cost_eur"] = (
        monthly["baseline_import_kwh"] * electricity_price
        - monthly["baseline_export_kwh"] * feed_in_tariff
    )
    monthly["post_battery_cost_eur"] = (
        monthly["post_battery_import_kwh"] * electricity_price
        - monthly["post_battery_export_kwh"] * feed_in_tariff
    )
    monthly["savings_eur"] = monthly["baseline_cost_eur"] - monthly["post_battery_cost_eur"]
    monthly.index = dispatch.index.strftime("%Y-%m")
    monthly.index.name = "month"
    return monthly.groupby(level=0, sort=True).sum()

def build_cash_flow_table(annual_gross_savings, assumptions):
    discount_rate = float(assumptions["discount_rate"])
    if discount_rate <= -1.0:
        raise ValueError("Discount rate must be greater than -100%")
    rows = [{
        "year": 0,
        "gross_savings_eur": 0.0,
        "om_cost_eur": 0.0,
        "net_cash_flow_eur": -float(assumptions["installed_cost"]),
        "discounted_cash_flow_eur": -float(assumptions["installed_cost"]),
    }]
    for year, gross_savings in enumerate(annual_gross_savings, start=1):
        om_cost = float(assumptions["annual_om_cost"])
        net_cash_flow = float(gross_savings) - om_cost
        rows.append({
            "year": year,
            "gross_savings_eur": float(gross_savings),
            "om_cost_eur": om_cost,
            "net_cash_flow_eur": net_cash_flow,
            "discounted_cash_flow_eur": net_cash_flow / (1.0 + discount_rate) ** year,
        })
    cash_flows = pd.DataFrame(rows).set_index("year")
    cash_flows["cumulative_cash_flow_eur"] = cash_flows["net_cash_flow_eur"].cumsum()
    cash_flows["cumulative_discounted_cash_flow_eur"] = cash_flows["discounted_cash_flow_eur"].cumsum()
    simple_payback = cash_flows.index[cash_flows["cumulative_cash_flow_eur"] >= 0]
    discounted_payback = cash_flows.index[cash_flows["cumulative_discounted_cash_flow_eur"] >= 0]
    cash_flows.attrs["npv_eur"] = float(cash_flows["discounted_cash_flow_eur"].sum())
    cash_flows.attrs["simple_payback_year"] = int(simple_payback[0]) if len(simple_payback) else None
    cash_flows.attrs["discounted_payback_year"] = int(discounted_payback[0]) if len(discounted_payback) else None
    return cash_flows

def build_lifecycle(counterfactual, assumptions, eligible_days, allow_partial_year):
    minimum_days = int(assumptions["minimum_days_for_lifecycle"])
    if eligible_days <= 0:
        return None, "Lifecycle economics unavailable: no eligible days remain."
    if eligible_days < minimum_days and not allow_partial_year:
        return None, (
            f"Lifecycle economics withheld: only {eligible_days} eligible days; "
            f"{minimum_days} are required. Partial-year annualization is disabled."
        )
    partial_year = eligible_days < minimum_days
    scaling_factor = 365.2425 / eligible_days if partial_year else 1.0
    annual_gross_savings = []
    capacities = []
    for year in range(1, int(assumptions["project_lifetime_years"]) + 1):
        capacity = float(assumptions["capacity_kwh"]) * (
            1.0 - float(assumptions["annual_capacity_degradation"])
        ) ** (year - 1)
        yearly_dispatch = simulate_battery(
            counterfactual,
            capacity_kwh=capacity,
            initial_soc_fraction=float(assumptions["initial_soc_fraction"]),
            max_charge_power_kw=float(assumptions["max_charge_power_kw"]),
            max_discharge_power_kw=float(assumptions["max_discharge_power_kw"]),
            charge_efficiency=float(assumptions["charge_efficiency"]),
            discharge_efficiency=float(assumptions["discharge_efficiency"]),
        )
        monthly_savings = calculate_observed_savings(
            yearly_dispatch,
            electricity_price=float(assumptions["electricity_price"]),
            feed_in_tariff=float(assumptions["feed_in_tariff"]),
        )
        annual_gross_savings.append(float(monthly_savings["savings_eur"].sum()) * scaling_factor)
        capacities.append(capacity)
    lifecycle = build_cash_flow_table(annual_gross_savings, assumptions)
    lifecycle["usable_capacity_kwh"] = [np.nan, *capacities]
    lifecycle.attrs["scaling_factor"] = scaling_factor
    lifecycle.attrs["partial_year"] = partial_year
    lifecycle.attrs["npv_eur"] = float(lifecycle["discounted_cash_flow_eur"].sum())
    lifecycle.attrs["simple_payback_year"] = int(
        lifecycle.index[lifecycle["cumulative_cash_flow_eur"] >= 0][0]
    ) if (lifecycle["cumulative_cash_flow_eur"] >= 0).any() else None
    lifecycle.attrs["discounted_payback_year"] = int(
        lifecycle.index[lifecycle["cumulative_discounted_cash_flow_eur"] >= 0][0]
    ) if (lifecycle["cumulative_discounted_cash_flow_eur"] >= 0).any() else None
    message = (
        "Measured-period dispatch repeated over the project lifetime; "
        f"annual gross savings scaling factor: {scaling_factor:.6g}."
    )
    return lifecycle, message

energy_summary_fixture = summarize_energy(dispatch_test)
assert np.isclose(energy_summary_fixture["baseline_import_kwh"], 6.0)
assert np.isclose(energy_summary_fixture["baseline_export_kwh"], 8.0)
assert np.isclose(
    energy_summary_fixture["post_net_grid_kwh"] - energy_summary_fixture["baseline_net_grid_kwh"],
    energy_summary_fixture["stored_energy_change_kwh"] + energy_summary_fixture["conversion_losses_kwh"],
)
assert abs(energy_summary_fixture["energy_balance_residual_kwh"]) < 1e-12

monthly_fixture = calculate_observed_savings(
    dispatch_test, electricity_price=0.30, feed_in_tariff=0.08
)
assert np.isclose(monthly_fixture["baseline_cost_eur"].sum(), 1.16)
assert np.isclose(monthly_fixture["savings_eur"].sum(), 0.9055555555555556)

cash_flow_fixture = build_cash_flow_table(
    [400.0],
    {"installed_cost": 1000.0, "annual_om_cost": 50.0, "discount_rate": 0.0},
)
assert np.isclose(cash_flow_fixture.loc[0, "net_cash_flow_eur"], -1000.0)
assert np.isclose(cash_flow_fixture.loc[1, "net_cash_flow_eur"], 350.0)

guarded_lifecycle, lifecycle_guard_message = build_lifecycle(
    counterfactual, ASSUMPTIONS, eligible_days=1, allow_partial_year=False
)
assert guarded_lifecycle is None
assert "eligible days" in lifecycle_guard_message
provisional_lifecycle, provisional_lifecycle_message = build_lifecycle(
    counterfactual, ASSUMPTIONS, eligible_days=1, allow_partial_year=True
)
assert provisional_lifecycle is not None
assert provisional_lifecycle.attrs["partial_year"]
assert np.isclose(provisional_lifecycle.attrs["scaling_factor"], 365.2425)
assert "scaling factor" in provisional_lifecycle_message


In [ ]:
eligible_days = int(daily_quality["eligible"].sum())
selected_summary = summarize_energy(dispatch)
monthly_results = calculate_observed_savings(
    dispatch,
    electricity_price=ELECTRICITY_PRICE_EUR_PER_KWH,
    feed_in_tariff=FEED_IN_TARIFF_EUR_PER_KWH,
)
lifecycle, lifecycle_message = build_lifecycle(
    counterfactual,
    ASSUMPTIONS,
    eligible_days=eligible_days,
    allow_partial_year=ALLOW_PARTIAL_YEAR_ANNUALIZATION,
)

assumptions_table = pd.Series({
    "Analysis start": ANALYSIS_START,
    "Analysis end (inclusive)": ANALYSIS_END or "latest available",
    "Resampling interval": RESAMPLE_FREQUENCY,
    "Minimum daily coverage": MINIMUM_DAILY_COVERAGE,
    "myPV thermal service": "space heating",
    "BWWP EL thermal service": "domestic hot water",
    "Shared heat-pump COP": HEAT_PUMP_COP,
    "Battery capacity (kWh)": BATTERY_CAPACITY_KWH,
    "Initial SOC fraction": INITIAL_SOC_FRACTION,
    "Charge / discharge limit (kW)": f"{MAX_CHARGE_POWER_KW:g} / {MAX_DISCHARGE_POWER_KW:g}",
    "Charge / discharge efficiency": f"{CHARGE_EFFICIENCY:.1%} / {DISCHARGE_EFFICIENCY:.1%}",
    "Annual capacity degradation": ANNUAL_CAPACITY_DEGRADATION,
    "Retail / feed-in price (EUR/kWh)": f"{ELECTRICITY_PRICE_EUR_PER_KWH:.2f} / {FEED_IN_TARIFF_EUR_PER_KWH:.2f}",
    "Installed cost (EUR)": INSTALLED_BATTERY_COST_EUR,
    "Annual O&M (EUR)": ANNUAL_OM_COST_EUR,
    "Project lifetime (years)": PROJECT_LIFETIME_YEARS,
    "Discount rate": DISCOUNT_RATE,
    "Eligible days / lifecycle minimum": f"{eligible_days} / {MINIMUM_DAYS_FOR_LIFECYCLE}",
    "Partial-year annualization": ALLOW_PARTIAL_YEAR_ANNUALIZATION,
}, name="value").to_frame()
print("Assumptions:")
display(assumptions_table)
print("Selected-scenario technical summary:")
display(selected_summary.to_frame("value"))
print("Observed-period monthly energy and cost:")
display(monthly_results)
if lifecycle is None:
    print(lifecycle_message)
else:
    print(lifecycle_message)
    if lifecycle.attrs["partial_year"]:
        print("PROVISIONAL: seasonally biased partial-year annualization; not suitable for an investment decision.")
    display(lifecycle)
    print(f"NPV: EUR {lifecycle.attrs['npv_eur']:.2f}")
    print(f"Simple payback year: {lifecycle.attrs['simple_payback_year']}")
    print(f"Discounted payback year: {lifecycle.attrs['discounted_payback_year']}")


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily_quality.index, daily_quality["coverage"] * 100.0, marker=".", linewidth=1)
ax.axhline(MINIMUM_DAILY_COVERAGE * 100.0, color="crimson", linestyle="--", label="eligibility threshold")
ax.set(title="Daily valid interval coverage", ylabel="Coverage (%)", xlabel="Local date")
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

eligible_date_set = set(daily_quality.index[daily_quality["eligible"]])
pv_energy_by_day = counterfactual["power_pv"].groupby(counterfactual.index.normalize()).sum() * interval_hours / 1000.0
three_day_candidates = []
for end_day in daily_quality.index[daily_quality["eligible"]]:
    candidate_days = pd.date_range(end_day - pd.Timedelta(days=2), end_day, freq="D")
    if all(day in eligible_date_set for day in candidate_days):
        three_day_candidates.append((float(pv_energy_by_day.reindex(candidate_days).sum()), candidate_days))
if three_day_candidates:
    selected_days = max(three_day_candidates, key=lambda item: item[0])[1]
else:
    eligible_dates = daily_quality.index[daily_quality["eligible"]]
    current_run = []
    longest_run = []
    for day in eligible_dates:
        if current_run and day != current_run[-1] + pd.DateOffset(days=1):
            if len(current_run) > len(longest_run):
                longest_run = current_run
            current_run = []
        current_run.append(day)
    if len(current_run) > len(longest_run):
        longest_run = current_run
    selected_days = pd.DatetimeIndex(longest_run[:3])
    print(f"No eligible three-day window; plotting the longest available run ({len(selected_days)} day(s)).")
representative = dispatch.loc[dispatch.index.normalize().isin(selected_days)]
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
axes[0].plot(representative.index, representative["power_pv"] / 1000.0, label="PV generation")
axes[0].plot(representative.index, representative["counterfactual_load"] / 1000.0, label="counterfactual load")
axes[0].set_ylabel("Power (kW)")
axes[0].set_title("Representative dispatch; import is positive and export is negative")
axes[0].legend(loc="upper right")
axes[0].grid(True, alpha=0.3)
axes[1].plot(representative.index, representative["counterfactual_mains"] / 1000.0, label="baseline mains")
axes[1].plot(representative.index, representative["post_battery_mains"] / 1000.0, label="post-battery mains")
axes[1].plot(representative.index, representative["battery_charge_power_w"] / 1000.0, label="battery charging")
axes[1].plot(representative.index, -representative["battery_discharge_power_w"] / 1000.0, label="battery discharging")
axes[1].axhline(0.0, color="black", linewidth=0.7)
axes[1].set_ylabel("Power (kW)")
axes[1].legend(loc="upper right", ncol=2)
axes[1].grid(True, alpha=0.3)
axes[2].plot(representative.index, representative["soc_kwh"], color="purple", label="SOC")
axes[2].set_ylabel("Stored energy (kWh)")
axes[2].set_xlabel("Local time")
axes[2].legend()
axes[2].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


In [ ]:
months = monthly_results.index.to_list()
x = np.arange(len(months))
width = 0.2
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].bar(x - 1.5 * width, monthly_results["baseline_import_kwh"], width, label="baseline import")
axes[0].bar(x - 0.5 * width, monthly_results["post_battery_import_kwh"], width, label="post-battery import")
axes[0].bar(x + 0.5 * width, monthly_results["baseline_export_kwh"], width, label="baseline export")
axes[0].bar(x + 1.5 * width, monthly_results["post_battery_export_kwh"], width, label="post-battery export")
axes[0].set_ylabel("Energy (kWh)")
axes[0].set_title("Monthly grid energy")
axes[0].legend(ncol=2)
axes[0].grid(True, axis="y", alpha=0.3)
axes[1].bar(x, monthly_results["savings_eur"], label="monthly observed savings")
axes[1].plot(x, monthly_results["savings_eur"].cumsum(), color="black", marker="o", label="cumulative observed savings")
axes[1].axhline(0.0, color="black", linewidth=0.7)
axes[1].set_ylabel("Value (EUR)")
axes[1].set_xlabel("Month")
axes[1].set_xticks(x, months, rotation=45)
axes[1].legend()
axes[1].grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

if lifecycle is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(lifecycle.index, lifecycle["cumulative_cash_flow_eur"], marker="o", label="undiscounted")
    ax.plot(lifecycle.index, lifecycle["cumulative_discounted_cash_flow_eur"], marker="o", label="discounted")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set(title="Lifecycle cumulative cash flow", xlabel="Project year", ylabel="Cumulative value (EUR)")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()


In [ ]:
sensitivity_rows = []
for capacity in SENSITIVITY_CAPACITIES_KWH:
    capacity_dispatch = simulate_battery(
        counterfactual,
        capacity_kwh=capacity,
        initial_soc_fraction=INITIAL_SOC_FRACTION,
        max_charge_power_kw=MAX_CHARGE_POWER_KW,
        max_discharge_power_kw=MAX_DISCHARGE_POWER_KW,
        charge_efficiency=CHARGE_EFFICIENCY,
        discharge_efficiency=DISCHARGE_EFFICIENCY,
    )
    capacity_savings = calculate_observed_savings(
        capacity_dispatch,
        electricity_price=ELECTRICITY_PRICE_EUR_PER_KWH,
        feed_in_tariff=FEED_IN_TARIFF_EUR_PER_KWH,
    )["savings_eur"].sum()
    capacity_npv = np.nan
    savings_label = "observed_period_savings_eur"
    if lifecycle is not None:
        capacity_assumptions = {**ASSUMPTIONS, "capacity_kwh": capacity}
        capacity_lifecycle, _ = build_lifecycle(
            counterfactual,
            capacity_assumptions,
            eligible_days=eligible_days,
            allow_partial_year=ALLOW_PARTIAL_YEAR_ANNUALIZATION,
        )
        capacity_savings = capacity_lifecycle.loc[1, "gross_savings_eur"]
        capacity_npv = capacity_lifecycle.attrs["npv_eur"]
        savings_label = "annual_gross_savings_eur"
    sensitivity_rows.append({
        "capacity_kwh": capacity,
        savings_label: float(capacity_savings),
        "equivalent_full_cycles": summarize_energy(capacity_dispatch)["equivalent_full_cycles"],
        "npv_eur": capacity_npv,
    })
sensitivity = pd.DataFrame(sensitivity_rows).set_index("capacity_kwh")
savings_column = "annual_gross_savings_eur" if lifecycle is not None else "observed_period_savings_eur"
sensitivity["marginal_savings_eur_per_added_kwh"] = (
    sensitivity[savings_column].diff() / sensitivity.index.to_series().diff()
)
display(sensitivity)
print("Capacity sensitivity keeps installed cost and charge/discharge power fixed; it is not a vendor cost curve.")

if sensitivity["npv_eur"].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(sensitivity.index, sensitivity[savings_column], marker="o")
    axes[0].set(ylabel="Annual gross savings (EUR)" if lifecycle is not None else "Observed-period savings (EUR)", xlabel="Capacity (kWh)")
    axes[1].plot(sensitivity.index, sensitivity["npv_eur"], marker="o", color="darkgreen")
    axes[1].set(ylabel="NPV (EUR)", xlabel="Capacity (kWh)")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sensitivity.index, sensitivity[savings_column], marker="o")
    ax.set(
        ylabel="Observed-period savings (EUR)",
        xlabel="Capacity (kWh)",
        title="Capacity sensitivity — observed period only",
    )
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    fig = None
if fig is not None:
    for axis in axes:
        axis.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


In [ ]:
assert replacement_energy_checks_pass
assert dispatch["soc_kwh"].between(-1e-9, BATTERY_CAPACITY_KWH + 1e-9).all()
assert not ((dispatch["battery_charge_power_w"] > 1e-9) & (dispatch["battery_discharge_power_w"] > 1e-9)).any()
assert maximum_dispatch_balance_residual_w < 1e-6
assert db_path.name in {"haslach.db", SNAPSHOT_FILENAME}
if eligible_days < MINIMUM_DAYS_FOR_LIFECYCLE and not ALLOW_PARTIAL_YEAR_ANNUALIZATION:
    assert lifecycle is None
print("All notebook validation checks passed.")
